# 05. Conclusions and draft Results text

Runs the whole pipeline in one call and applies the pre-set decision rules (`pipeline/conclude.py`, plan step 7). **Dummy data by default: none of this may be cited until `DATA_DIR` points at real data.**

Rules: H1 to H3 are the *null* hypotheses (proposal 4.1). A null is rejected only when the omnibus test of the decision source (the primary model, or the Friedman fallback when the primary is flagged) has p < .05; otherwise we *fail to reject* (never 'no difference'). Pairwise contrasts are reported only after a significant omnibus test and only if they survive Holm correction.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)

DATA_DIR = ROOT / "data" / "dummy"      # <- point this at data/raw/<your export folder> for real data
OUT_DIR = ROOT / "outputs"

from IPython.display import Markdown, display
from pipeline.run import run_all
from pipeline.conclude import mechanism

res = run_all(DATA_DIR, OUT_DIR)
print(f"{len(res.prep.trials)} trials, {res.prep.trials.participant_id.nunique()} participants -> {OUT_DIR}")

89 trials, 15 participants -> c:\Users\User\Documents\AY26-27\HCI2000\outputs


## Hypothesis decisions

In [2]:
res.decisions[['hypothesis','outcome','test','statistic','df','p','decision','significant_contrasts','caution']].round(4)

,hypothesis,outcome,test,statistic,df,p,decision,significant_contrasts,caution
0,H1,T_find,linear mixed model on log T_find,28.3534,2,0.0000,Reject H1,MM vs DH; ND vs DH; ND vs MM,
1,H2,E,Poisson GEE (dispersion 1.16),11.5116,2,0.0032,Reject H2,ND vs MM,
2,H3,R,linear mixed model on log R,8.5029,2,0.0142,Reject H3,ND vs MM,unequal residual spread across patterns (Leven...


## Where does the time go? (RQ3)
The gap in mean T_find between two patterns splits exactly into a recovery part and a clean-navigation part.

In [3]:
tab, verdict = mechanism(res.prep, res.suite)
print(verdict)
tab.round(2)

For the contrasts that differ (MM vs DH, ND vs DH, ND vs MM), recovery accounts for 30% of the absolute gap in mean time; clean navigation time itself also differs across patterns (Wald χ²(2) = 44.93, p < .001). The differences are therefore mainly clean navigation (the time spent scanning and making correct selections), not recovery.


,contrast,delta_T_find,delta_T_clean,delta_R,share_recovery
0,MM vs DH,-3.03,-2.15,-0.87,0.29
1,ND vs DH,2.66,1.83,0.83,0.31
2,ND vs MM,5.69,3.98,1.71,0.30


## Limitations checklist
`check` items need discussion in the paper; `note` items are standing design caveats.

In [4]:
res.limitations

,item,status,detail
0,Sample size,note,15 participants analysed (planned 15); the siz...
1,Failed trials,flag,"Give-up/time-out trials by pattern: DH 2, MM 2..."
2,Fallback tests used,flag,"Decision uses the Friedman test for: success, ..."
3,Model assumptions,flag,Assumption flags with no usable fallback for: R.
4,Recovery-time sample,note,"Trials with at least one error: DH 8, MM 9, ND..."
5,Sensitivity analyses,flag,8 of 10 analyses (including the primary) agree...
6,Multiple hypotheses,note,H1 to H3 are each tested at α = .05 with no co...
7,Population-averaged effects,note,Error and success effects come from GEE and ar...
8,Block position and task pair,flag,These are the same factor in the Latin square ...
9,Minimum selections,note,The mega-menu needs 2 selections and the other...


## Draft Results and Discussion text
Also saved to `outputs/results_draft.md`. It is a starting point: verify every number against the tables and write the interpretation yourselves.

In [5]:
display(Markdown(res.draft_path.read_text(encoding='utf-8')))

# Results and Discussion: auto-generated draft

> **SYNTHETIC DATA.** This draft was generated from dummy data to test the pipeline. None of it is a finding and none of it may be cited.

> Starting text only: check every number against the tables, and write the interpretation yourselves. Generated by `pipeline/conclude.py`.

## Participants and trials

15 participants (age 31.1 ± 6.9 years, range 20–40) completed 89 analysed experimental trials (85 successful). 7 had used the reference website before; mean familiarity with government websites was 2.80 on a 1–5 scale. Practice trials and 1 trial(s) excluded for device or app faults were removed before analysis.

## Task success

Success rates were Deep hierarchical 93% (28/30), Mega-menu 93% (27/29), and Nested dropdown 100% (30/30). Failures were give-ups (1) and time-outs (3). 

Success rates did not differ across patterns (Friedman χ²(2) = 2.00, p = .368, Kendall's W = 0.07, n = 15 participants; the primary model was flagged: primary model not estimable: complete separation: every trial in pattern(s) ND has the same outcome, so odds ratios are not estimable). Failed trials are excluded from the time and error analyses below.

## Hypothesis tests

### H1: item-finding time

Median item-finding time was Deep hierarchical 9.5 s (IQR 8.0–11.6), Mega-menu 7.4 s (IQR 6.0–10.0), and Nested dropdown 13.1 s (IQR 9.7–16.7) (successful trials).

Item-finding time differed across navigation patterns (Wald χ²(2) = 28.35, p < .001). We tested this with a linear mixed model on log T_find.

After Holm correction: Mega-menu item-finding time was 23% shorter than Deep hierarchical (ratio of geometric means 0.77, 95% CI [0.64, 0.93]; Holm-adjusted p = .015); Nested dropdown item-finding time was 27% longer than Deep hierarchical (ratio of geometric means 1.27, 95% CI [1.06, 1.53]; Holm-adjusted p = .015); Nested dropdown item-finding time was 65% longer than Mega-menu (ratio of geometric means 1.65, 95% CI [1.37, 1.98]; Holm-adjusted p < .001).

We therefore reject the null hypothesis H1 in favour of H1a.

### H2: error count

Mean errors per trial were Deep hierarchical 0.50 (SD 0.92), Mega-menu 0.33 (SD 0.48), and Nested dropdown 0.90 (SD 1.09); the share of trials with at least one error was Deep hierarchical 29%, Mega-menu 33%, and Nested dropdown 47%.

Error count differed across navigation patterns (Wald χ²(2) = 11.51, p = .003). We tested this with a Poisson GEE (dispersion 1.16).

After Holm correction: Nested dropdown trials had 2.6 times as many errors as Mega-menu (rate ratio 2.60, 95% CI [1.39, 4.86]; Holm-adjusted p = .008).

We therefore reject the null hypothesis H2 in favour of H2a.

### H3: recovery time

Among trials with at least one error, median recovery time was Deep hierarchical 4.8 s (IQR 2.5–9.1; n = 8 trials), Mega-menu 2.6 s (IQR 2.3–3.0; n = 9 trials), and Nested dropdown 4.8 s (IQR 4.0–6.9; n = 14 trials).

Recovery time differed across navigation patterns (Wald χ²(2) = 8.50, p = .014). We tested this with a linear mixed model on log R.

After Holm correction: Nested dropdown recovery time was 91% longer than Mega-menu (ratio of geometric means 1.91, 95% CI [1.22, 3.00]; Holm-adjusted p = .014).

We therefore reject the null hypothesis H3 in favour of H3a.

*Caution: unequal residual spread across patterns (Levene p = 0.0039); fallback also unavailable: only 3 participants have a value under all three patterns.*

### Where does the time go? (RQ3, Eq. 2)

For the contrasts that differ (MM vs DH, ND vs DH, ND vs MM), recovery accounts for 30% of the absolute gap in mean time; clean navigation time itself also differs across patterns (Wald χ²(2) = 44.93, p < .001). The differences are therefore mainly clean navigation (the time spent scanning and making correct selections), not recovery.

## Secondary outcomes

Mean perceived difficulty (Single Ease Question, 1 = very difficult, 7 = very easy) was Deep hierarchical 5.07, Mega-menu 5.69, and Nested dropdown 5.53; ratings did not differ across patterns (Friedman χ²(2) = 3.17, p = .205, Kendall's W = 0.11, n = 15 participants). 

Mean preference rank (1 = most preferred) was Deep hierarchical 2.47, Mega-menu 1.80, and Nested dropdown 1.73; first choices: Deep hierarchical 2, Mega-menu 7, and Nested dropdown 6 (Friedman χ²(2) = 4.93, p = .085). These outcomes support, rather than test, the hypotheses.

## Robustness

8 of 10 analyses agree with the primary significance (primary models, excluding block 1, adding covariates, and a Cox model treating failures as censored). Disagreements: E (excluding block position 1); R (excluding block position 1).

## Limitations to discuss

- **Sample size** (note): 15 participants analysed (planned 15); the size is provisional pending the pilot power analysis. A non-significant result means 'not detected', not 'no difference'.
- **Failed trials** (check): Give-up/time-out trials by pattern: DH 2, MM 2, ND 0. They are excluded from the time and error models (proposal 4.7), which can understate the slowest patterns; see the censored sensitivity analysis.
- **Fallback tests used** (check): Decision uses the Friedman test for: success, SEQ (reasons in model_summary.csv). Friedman has less power and gives no covariate adjustment.
- **Model assumptions** (check): Assumption flags with no usable fallback for: R.
- **Recovery-time sample** (note): Trials with at least one error: DH 8, MM 9, ND 14. Recovery time (H3) rests only on these trials, so its estimates are less stable.
- **Sensitivity analyses** (check): 8 of 10 analyses (including the primary) agree with the primary significance. Disagreeing: E (excluding block position 1); R (excluding block position 1).
- **Multiple hypotheses** (note): H1 to H3 are each tested at α = .05 with no correction across the three hypotheses (Holm is applied only to pairwise contrasts within each).
- **Population-averaged effects** (note): Error and success effects come from GEE and are population-averaged rather than participant-specific (Table 4 proposed mixed models).
- **Block position and task pair** (check): These are the same factor in the Latin square (Table 3), so a block effect cannot be separated from task-pair difficulty.
- **Minimum selections** (note): The mega-menu needs 2 selections and the other patterns 3; this is part of what the pattern is (proposal 4.3), not removed.
- **Generalisability** (note): One reference website, one device model, and a convenience sample (friends and family recruited via Facebook/Messenger).

## Tables

### Hypothesis decisions

| hypothesis | outcome | test | statistic | df | p | decision | significant_contrasts |
| --- | --- | --- | --- | --- | --- | --- | --- |
| H1 | T_find | linear mixed model on log T_find | 28.353 | 2 | 0.000 | Reject H1 | MM vs DH; ND vs DH; ND vs MM |
| H2 | E | Poisson GEE (dispersion 1.16) | 11.512 | 2 | 0.003 | Reject H2 | ND vs MM |
| H3 | R | linear mixed model on log R | 8.503 | 2 | 0.014 | Reject H3 | ND vs MM |

### Per-pattern summary (successful trials unless stated)

| pattern | n_trials | success_rate | T_find_median | E_mean | share_E_ge1 | n_trials_E_ge1 | R_median | SEQ_mean |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| DH | 30 | 0.93 | 9.50 | 0.50 | 0.29 | 8 | 4.76 | 5.07 |
| MM | 29 | 0.93 | 7.39 | 0.33 | 0.33 | 9 | 2.57 | 5.69 |
| ND | 30 | 1.00 | 13.14 | 0.90 | 0.47 | 14 | 4.76 | 5.53 |

## Figures

F1 T_find by pattern (H1); F2 errors (H2); F3 recovery time (H3); F4 clean navigation vs recovery (RQ3); F5 success; F6 difficulty ratings; F7 preference; F8 block/task pair check; F9-F10 path analyses; F11 model diagnostics; F12 pairwise effect sizes. All in `outputs/figures/`, with the data behind each in `outputs/tables/fig_data_*.csv`.
